# 00 — Setup and smoke test

**Purpose.** Prove the whole pipeline works on a fresh Colab runtime: GPU, Drive, pinned
install, unit tests, released LeWM checkpoints + datasets (TwoRoom, PushT), and a
2-episode run of the *official* LeWM evaluation (`third_party/le-wm/eval.py`, unmodified).

**Expected runtime.** ~15 min with `ENVS = ["tworoom"]`, ~30 min with PushT as well
(dominated by the 3.4 GB + 13.1 GB dataset downloads). Not yet measured — this run measures it.

**Expected compute.** ≤ 0.5 L4-hours (logged to the compute ledger).

**How to run.** Runtime → Change runtime type → L4 GPU (or A100). Edit the config cell
below (only `REPO_URL` is mandatory), then Runtime → Run all. If pip asks for a runtime
restart, restart and Run all again: every step is idempotent.

## 1. Config (the only cell you edit)

In [ ]:
# --- repository -------------------------------------------------------------
REPO_URL = "https://github.com/CHANGE_ME/jepa-memory.git"  # <-- your fork / remote
REPO_REF = "main"            # branch, tag or commit to test
REPO_DIR = "/content/jepa-memory"
# Private repo: add a Colab secret named GH_TOKEN (key icon in the left bar).

# --- what to run ------------------------------------------------------------
ENVS = ["tworoom", "pusht"]  # subset of: tworoom (3.4 GB), pusht (13.1 GB)
SMOKE_EPISODES = 2           # official protocol uses 50; 2 is only a pipeline check
SEED = 42                    # official evaluation seed
INSTALL_EXTRA = "wm,dev"     # "wm" = 2-D envs; "wm-all,dev" adds MuJoCo envs (Reacher/Cube)
FORCE = False                # True: re-run evaluations whose JSON already exists

# --- bookkeeping ------------------------------------------------------------
RUN_ID = "nb00-smoke"
NOTEBOOK = "00_setup_and_smoke"
EST_HOURS = 0.5              # estimate stated before the run (ledger)
EXPECTED_GPUS = ("L4", "A100")

## 2. GPU check

In [ ]:
import subprocess

try:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu_name = ""
print("GPU:", gpu_name or "NONE")
HAS_GPU = bool(gpu_name)
if not any(g in gpu_name for g in EXPECTED_GPUS):
    print(f"WARNING: expected one of {EXPECTED_GPUS}. The official eval script needs CUDA "
          "(le-wm/eval.py:89); budget accounting assumes L4 or A100.")

## 3. Drive mount

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not on Colab: using repo-relative paths.")

## 4. Clone / update the repo (with the pinned `le-wm` submodule)

In [ ]:
def _git(*args, cwd=None):
    subprocess.run(["git", *args], cwd=cwd, check=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    assert "CHANGE_ME" not in REPO_URL, "Set REPO_URL in the config cell."
    clone_url = REPO_URL
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
            if token:
                clone_url = REPO_URL.replace("https://", f"https://{token}@")
        except Exception:
            pass  # no secret configured -> public clone
    subprocess.run(["git", "clone", "-q", clone_url, REPO_DIR], check=True, capture_output=True)
    _git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)  # never keep the token on disk
else:
    _git("fetch", "-q", "--all", cwd=REPO_DIR)
_git("checkout", "-q", REPO_REF, cwd=REPO_DIR)
on_branch = subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO_DIR,
                           capture_output=True).returncode == 0
if on_branch:
    subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO_DIR)  # ok to fail offline
_git("submodule", "update", "--init", "--recursive", cwd=REPO_DIR)
os.chdir(REPO_DIR)
print("repo at", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["git", "submodule", "status"], capture_output=True, text=True).stdout.strip())

## 5. Install pinned dependencies and the `jmem` package

In [ ]:
# torch is NOT reinstalled: Colab's CUDA build satisfies torch>=2.4,<3 (see pyproject.toml).
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f".[{INSTALL_EXTRA}]"])
assert r.returncode == 0, "pip install failed (scroll up for the error)"
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)  # editable installs are only picked up by new processes
import jmem
print("jmem", jmem.__version__)

## 6. Paths, seeds, ledger

In [ ]:
from jmem import paths, ledger, utils, datasets
from jmem.evaluate import run_official_eval, check_protocol, load_protocol

P = paths.setup()
utils.seed_everything(SEED)
ledger.start(RUN_ID, est_hours=EST_HOURS, notebook=NOTEBOOK)

for k, v in P.as_dict().items():
    print(f"{k:12s} {v}")
versions = utils.env_versions()
print(versions)

import datetime
lock = P.locks / f"pip-freeze-{datetime.date.today().isoformat()}.txt"
lock.write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                               capture_output=True, text=True).stdout)
print("lock file:", lock)

## 7. Unit tests (`pytest -m 'not slow'`)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-m", "not slow", "-q"],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-1000:])
TESTS_OK = r.returncode == 0
assert TESTS_OK, "unit tests failed"

## 8. Released checkpoints and datasets

In [ ]:
import json

DATA = {}
for env in ENVS:
    ckpt_dir = datasets.ensure_checkpoint(env)
    h5 = datasets.ensure_dataset(env, P.data_home)
    DATA[env] = {"checkpoint_dir": str(ckpt_dir),
                 "checkpoint_files": sorted(p.name for p in ckpt_dir.iterdir()),
                 "dataset": datasets.describe_h5(h5)}
    print(json.dumps(DATA[env], indent=1))
(P.results / "smoke").mkdir(parents=True, exist_ok=True)
(P.results / "smoke" / "data_inventory.json").write_text(json.dumps(DATA, indent=2))

## 9. Smoke evaluation: official `eval.py`, 2 episodes per env

This calls `third_party/le-wm/eval.py` unchanged (through `jmem._official_eval_runner`,
which only records the arguments and metrics of the script's single `World.evaluate` call).
Two episodes say nothing about success rates; they only prove the pipeline runs.

In [ ]:
protocol = load_protocol()
RECORDS, MISMATCHES = {}, {}
if not HAS_GPU:
    print("SKIPPED: no CUDA device; le-wm/eval.py hard-codes model.to('cuda').")
else:
    for env in ENVS:
        rec = run_official_eval(env, datasets.CHECKPOINTS[env], out_dir=P.results / "smoke",
                                num_eval=SMOKE_EPISODES, seed=SEED, paths=P, force=FORCE)
        RECORDS[env] = rec
        # num_eval is deliberately 2 here; everything else must equal configs/eval/protocol.yaml
        MISMATCHES[env] = check_protocol(rec, protocol, env, ignore=["num_eval"])
        print(env, "success_rate(%) =", rec["metrics"]["success_rate"],
              "| wallclock_s =", round(rec["wallclock_s"], 1),
              "| protocol mismatches =", MISMATCHES[env])

## 10. Validation

In [ ]:
import math

assert TESTS_OK
for env in ENVS:
    assert datasets.dataset_path(env, P.data_home).exists(), f"{env}: dataset missing"
    assert "weights.pt" in DATA[env]["checkpoint_files"] and "config.json" in DATA[env]["checkpoint_files"]
    assert DATA[env]["dataset"]["n_episodes"] > 0
if HAS_GPU:
    for env in ENVS:
        rec = RECORDS[env]
        path = P.results / "smoke" / f"official_{env}_n{SMOKE_EPISODES}_s{SEED}.json"
        assert path.exists(), path
        json.loads(path.read_text())  # valid JSON
        sr = rec["metrics"]["success_rate"]
        assert isinstance(sr, float) and not math.isnan(sr) and 0.0 <= sr <= 100.0, sr
        assert len(rec["metrics"]["episode_successes"]) == SMOKE_EPISODES
        assert rec["num_envs"] == SMOKE_EPISODES
        assert MISMATCHES[env] == [], MISMATCHES[env]
    print("all validation checks passed")
else:
    print("validation of the evaluation step SKIPPED (no GPU) - Phase 0 acceptance needs a GPU run")

## 11. Ledger and summary (paste everything below back to Claude)

In [ ]:
row = ledger.stop()
budget = ledger.summary()

print("\n===== NB00 SUMMARY (paste back) =====")
print("repo_sha      :", utils.git_sha())
print("gpu           :", gpu_name)
print("versions      :", json.dumps(versions))
print("tests_ok      :", TESTS_OK)
print("ledger_row    :", json.dumps(row))
print("ledger_used_L4h / projected:", round(budget.used, 3), "/", round(budget.projected, 3))
for env in ENVS:
    d = DATA[env]["dataset"]
    print(f"--- {env}")
    print("  dataset     :", d["n_episodes"], "episodes,", d["n_steps"], "steps, ep_len min/max", d["ep_len_min_max"])
    print("  columns     :", {k: v["shape"] for k, v in d["columns"].items()})
    if env in RECORDS:
        rec = RECORDS[env]
        print("  success_rate:", rec["metrics"]["success_rate"], "% over", rec["num_envs"], "episodes (smoke only)")
        print("  successes   :", rec["metrics"]["episode_successes"])
        print("  start_steps :", rec["evaluate_kwargs"].get("start_steps"))
        print("  episodes    :", rec["evaluate_kwargs"].get("episodes_idx"))
        print("  pixels shape:", rec["info_pixels_shape"], "(dim 1 = real frames the planner sees)")
        print("  policy      :", json.dumps(rec["policy"]))
        print("  eval_wall_s :", round(rec["evaluate_wallclock_s"], 1), "| total_wall_s:", round(rec["wallclock_s"], 1))
        print("  mismatches  :", MISMATCHES[env])